# Introduction to data-space inversion (DSI)

Reactive-transport models are expensive. A single DIZON run costs roughly 6 minutes on a laptop, and an honest decision-support analysis wants hundreds of runs. That cost is the wall this whole curriculum is built to get over. Data-space inversion (DSI) is the method we lean on most heavily to get over it, so it earns its own background notebook.

This is a *standalone* part0 notebook: no DIZON outputs are needed, nothing here writes a workspace, and you can read it in any order with the other part0 notebooks. It builds the intuition; the working version on the real model lives in [part1_05_dsi_basics](../part1_05_dsi_basics/dizon_dsi_basics.ipynb).

### Where this sits in the sequence

By now you have met the prior, the ensemble, and the idea of a Bayesian posterior (in [part0_03_uq_for_rtm](../part0_03_uq_for_rtm/uq_for_rtm.ipynb)), and you have seen how PEST++ and the iterative ensemble smoother (PESTPP-IES) condition a model on data (in [part0_04_intro_to_pest_and_ies](../part0_04_intro_to_pest_and_ies/intro_to_pest_and_ies.ipynb)). All of those workflows share one assumption: that you can afford to run the model many times, inside an iterative loop.

For DIZON, we mostly cannot. DSI is the answer. The idea is deceptively simple: run the model a few hundred times *once* (a prior Monte Carlo), then build a cheap stand-in - an "emulator" - that learns the statistical relationship between what we *measured* and what we want to *forecast*, and do all the conditioning on that stand-in instead. The stand-in runs in milliseconds.

This notebook is adapted from the GMDSI [`intro_to_eva_and_dsi`](https://github.com/gmdsi/GMDSI_notebooks) notebook. We keep its toy example - it is the clearest way to see the machinery - but we frame it around the DIZON forecast throughout.

### Admin

There is nothing expensive here. We need only `numpy`, `pandas`, and `matplotlib` for the toy demonstration, plus `pyemu` so we can point at the production `DSI` class when the time comes. We assert that `pyemu` is the vendored copy from `dependencies/`, the same check every notebook in this series does.

In [ ]:
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import pyemu
assert "dependencies" in pyemu.__file__
sys.path.insert(0, "..")

## The one idea: work in observation space, not parameter space

Classical history matching estimates *parameters*. For DIZON that means hydraulic conductivity fields, porosity, dispersivity, pyrite abundance and pyrite oxidation rate (the [layered prior](../../CONTEXT.md)) - thousands of adjustable numbers once you pilot-point a 3D field. Every one of those numbers is a dimension the algorithm has to search, and every step of the search costs a batch of 6-minute model runs.

But notice what we actually *want*. We do not care about the hydraulic conductivity in cell (42, 17, 5) for its own sake. We care about one forecast: the peak sulfate concentration at the supply well (`wellopt`) during the supply period — its distribution sets the treatment capacity the operator must design for. And we condition on a modest set of measurements: history-period breakthrough of SO₄, O₂, NO₃, pH and temperature at the monitoring sites.

Both of those - the measurements and the forecast - are *model outputs*. They live in **observation space**. DSI's move is to never leave observation space. Instead of asking "what parameters reproduce the data, and what do those parameters then predict?", it asks the question directly: "given the statistical relationship between measurements and forecast across the prior ensemble, what does the forecast look like once we pin the measurements to their observed values?"

The dimensionality of observation space is set by the number of outputs we track - hundreds - not by the number of parameters - thousands or more. That is the trade we are making, and it is why DSI is cheap.

## A toy ensemble of model outputs

To see the machinery without waiting on the model, we will cook up a fake ensemble of outputs. Imagine a model that produces three history-period "measurements" and one "forecast". Read the three measurements as, say, history-period SO₄ at three monitoring filters, and the forecast as peak SO₄ at the supply well. We pretend we have already run the prior Monte Carlo: 1000 realisations, each giving one value of each output.

In the real workflow these 1000 rows would be the (thinned) prior-MC observation ensemble - the expensive part, paid once. Here we draw them from a known multivariate normal so we can check our arithmetic against the truth at the back of the book:

In [ ]:
# means of [forecast, meas1, meas2, meas3]
mean = [0, 1, 2, 3]

# the (hidden) true covariance - the answer at the back of the book
true_cov = [
    [1.0, 0.8, 0.5, 0.5],
    [0.8, 1.0, 0.3, 0.3],
    [0.5, 0.3, 1.0, 0.2],
    [0.5, 0.3, 0.2, 1.0],
]

# number of realisations in our pretend prior Monte Carlo
nreal = 1000

# the prior observation ensemble - normally this costs nreal x 6 minutes of model runs
np.random.seed(42)
prior_oe = pd.DataFrame(
    np.random.multivariate_normal(mean, true_cov, nreal),
    columns=["forecast", "meas1", "meas2", "meas3"],
)
prior_oe.head()

Every pair of outputs is correlated - that correlation is the whole point. The forecast is not independent of the measurements; knowing the measurements tells us something about the forecast. A scatter matrix makes the relationships visible:

In [ ]:
_ = pd.plotting.scatter_matrix(prior_oe, figsize=(7, 7))

The off-diagonal clouds lean. `meas1` and `forecast` are tightly correlated (we set that correlation to 0.8); `meas3` and `forecast` less so. DSI is going to harvest exactly this structure - the cross-covariance between what we measure and what we forecast - to update the forecast when we learn the measured values.

## The empirical covariance $\mathbf{C}_d$

Collect all the outputs into one vector $\mathbf{d}$ (measurements *and* forecast together). The ensemble gives us a sample estimate of its covariance:

$$
\mathbf{C}_d = \frac{1}{N_e - 1} \sum_{i=1}^{N_e} (\mathbf{d}_i - \bar{\mathbf{d}})(\mathbf{d}_i - \bar{\mathbf{d}})^{\mathsf{T}}
$$

In words: take each realisation's output vector, subtract the ensemble mean $\bar{\mathbf{d}}$, form the outer product with itself, average over the ensemble. $N_e$ is the ensemble size. $\mathbf{C}_d$ holds the variance of every output on its diagonal and the covariance between every pair off-diagonal. It is the only thing DSI learns from the model - everything downstream is built from $\mathbf{C}_d$ and $\bar{\mathbf{d}}$.

In [ ]:
# the ensemble mean vector, d_bar
d_bar = prior_oe.mean()
d_bar

In [ ]:
# the empirical covariance - an approximation of true_cov above
Cd = prior_oe.cov()
Cd

Compare `Cd` to `true_cov`: with 1000 realisations the empirical estimate is close, but not exact. That gap matters. The whole emulator is built on $\mathbf{C}_d$, so the quality of the emulator is capped by how well a *finite* ensemble estimates the covariance. Too few realisations and the cross-covariances are noisy, and the forecast update is unreliable. This is one of the practical limits we return to at the end.

## From covariance to a latent space: the SVD

DSI does not use $\mathbf{C}_d$ directly. It needs its square root, $\mathbf{C}_d^{1/2}$, and it gets there through the singular value decomposition (SVD) of the *deviations matrix*. The deviations matrix $\Delta\mathbf{D}$ is just the ensemble with the mean removed and scaled so that $\Delta\mathbf{D}\,\Delta\mathbf{D}^{\mathsf{T}} = \mathbf{C}_d$:

$$
\Delta\mathbf{D} = \frac{1}{\sqrt{N_e - 1}}\left[\,\mathbf{d}_1 - \bar{\mathbf{d}},\ \ldots,\ \mathbf{d}_{N_e} - \bar{\mathbf{d}}\,\right]
$$

(To match the notation in the source papers we transpose so rows are outputs and columns are realisations.)

In [ ]:
# deviations matrix, shape (n_outputs, n_reals)
deltaD = prior_oe.T.apply(
    lambda row: (row - row.mean()) / np.sqrt(prior_oe.shape[0] - 1),
    axis=1,
)
deltaD.shape

Now take the SVD (see the GMDSI intro-to-SVD notebook if singular value decomposition is new):

$$
\Delta\mathbf{D} = \mathbf{U}\,\mathbf{\Sigma}\,\mathbf{V}^{\mathsf{T}}
$$

$\mathbf{U}$ and $\mathbf{V}$ are orthogonal - their columns are orthonormal directions - and $\mathbf{\Sigma}$ is diagonal, holding the singular values in decreasing order. The columns of $\mathbf{U}$ are the *principal directions* of the output ensemble: combinations of outputs that vary together. The singular values say how much of the ensemble's variability each direction carries.

In [ ]:
U, Sigma, Vt = np.linalg.svd(deltaD, full_matrices=False)
U.shape, Sigma.shape, Vt.shape

From the SVD, the square root of the covariance is just:

$$
\mathbf{C}_d^{1/2} = \mathbf{U}\,\mathbf{\Sigma}
$$

This is the **latent space**. The output ensemble - however many outputs we track - is summarised by a handful of orthogonal directions. With four toy outputs there are at most four; with the real DIZON obs set there are hundreds of *possible* directions, but the singular values fall off fast and we keep only enough to explain, say, 97.5% of the variance. That truncation is what makes DSI's "parameters" few, even when the model's parameters are legion.

In [ ]:
Cd_sqrt = U @ np.diag(Sigma)
Cd_sqrt.shape

## The emulator is a one-line forward model

Here is the whole emulator. Any plausible output vector $\mathbf{d}_{\text{PCA}}$ can be written as the mean plus a step in latent space:

$$
\mathbf{d}_{\text{PCA}} = \bar{\mathbf{d}} + \mathbf{C}_d^{1/2}\,\mathbf{x}
$$

where $\mathbf{x}$ is a vector of latent-space coordinates. "Running the emulator" means picking an $\mathbf{x}$ and evaluating that line. No PDE, no chemistry, no 6-minute wait - a matrix-vector product.

The vector $\mathbf{x}$ holds **latent-space parameters**. These are *not* the model's physical parameters. You can think of them as super-parameters that the SVD distilled out of the prior ensemble - a coordinate system in which the outputs are uncorrelated, standard-normal, and few. When we put DSI to work, these are the numbers PEST++ adjusts.

A "forward run" of the emulator, then, is one function. We start every latent parameter at zero, which - because the latent coordinates are centred on the mean - just returns the ensemble mean back:

In [ ]:
def emulator_forward_run(x):
    """The entire DSI emulator: mean plus a step in latent space."""
    return d_bar.values + Cd_sqrt @ x

x = np.zeros_like(Sigma)        # prior mean of the latent parameters
emulator_forward_run(x), d_bar.values

### "But we haven't done anything?"

Correct - with all latent parameters at zero we get the prior mean back, which is no use as a forecast. The payoff comes from *changing* $\mathbf{x}$. Different $\mathbf{x}$ vectors map to different, self-consistent output vectors that all honour the prior covariance structure. The job of conditioning is to find the values of $\mathbf{x}$ that make the emulator's *measurement* outputs match the data we actually observed - and then read off whatever the emulator says about the *forecast*.

This is exactly a calibration problem, except the forward model is the one-liner above instead of DIZON. That is the trick in one sentence.

### One caveat: the toy is Gaussian, the real model is not

In this toy example the emulator is *exactly* linear, because we drew the ensemble from a Gaussian. Real RTM outputs are not Gaussian: concentrations are non-negative and bounded, breakthrough curves are skewed. Run the linear emulator on untransformed concentrations and it will cheerfully predict negative SO₄.

The production `pyemu.emulators.DSI` handles this by applying a **transform** to the outputs *before* the SVD - a log, a standard-scaler, or a normal-score transform - so the emulator is linear in transformed space and the back-transform keeps predictions physical. Choosing that transform is a genuine beat in the real workflow (we deliberately let the untransformed version fail first in [part1_05_dsi_basics](../part1_05_dsi_basics/dizon_dsi_basics.ipynb), then fix it). The available transforms in this `pyemu` build are:

In [ ]:
# the transform "menu" the production DSI class understands
[t for t in ["log10", "standard_scaler", "normal_score", "row_wise_minmax"]]

## Conditioning the emulator

How do we find the $\mathbf{x}$ that honours the data? In the real workflow we hand the emulator to PESTPP-IES and let the iterative ensemble smoother do it, with a noise ensemble and `ies_multimodal_alpha=0.99` (explained in [part1_05_dsi_basics](../part1_05_dsi_basics/dizon_dsi_basics.ipynb)). But because each emulator run is so cheap, we can show the idea with brute force here: **rejection sampling**.

The recipe: draw a huge cloud of latent-parameter vectors, run every one through the emulator (it costs nothing), keep only the ones whose *measurement* outputs land close to the truth, and look at what those survivors say about the forecast. That is pure Bayesian conditioning - no gradients, no iterations.

First, choose a "truth". We borrow one realisation from the prior ensemble and treat its measurements as the data we observed and its forecast as the value we are trying to recover:

In [ ]:
truth = prior_oe.iloc[-1]
truth

Now draw a large prior ensemble of latent parameters. These are standard-normal by construction - independent, mean zero, variance one - which is the prior on $\mathbf{x}$:

In [ ]:
num_reals = 10000
prior_x = np.random.standard_normal((num_reals, Sigma.shape[0]))
prior_x.shape

Run them all through the emulator. Ten thousand "model runs" in the blink of an eye - the same ten thousand would be six weeks of DIZON runs:

In [ ]:
emu_prior = pd.DataFrame(
    [emulator_forward_run(xi) for xi in prior_x],
    columns=prior_oe.columns,
)
emu_prior.head()

Score each realisation by how well its *measurements* (not its forecast - we never get to see the truth's forecast) reproduce the observed measurements. We use a plain sum of squared residuals, the same currency PEST calls "phi", assuming unit weights:

In [ ]:
meas_cols = ["meas1", "meas2", "meas3"]
emu_prior["phi"] = ((emu_prior[meas_cols] - truth[meas_cols]) ** 2).sum(axis=1)
_ = plt.hist(emu_prior["phi"], facecolor="0.5", alpha=0.6, bins=40)
plt.xlabel("phi (sum of squared measurement residuals)")
plt.ylabel("count")

Keep only the "behavioural" realisations. A defensible threshold is phi less than or equal to the number of measurements - realisations whose average squared residual is within the noise:

In [ ]:
n_meas = len(meas_cols)
emu_post = emu_prior.loc[emu_prior["phi"] <= n_meas]
emu_post.shape[0], "behavioural realisations out of", num_reals

The payoff. Compare the forecast distribution before conditioning (prior) and after (posterior), with the truth's actual forecast marked. If DSI worked, the posterior is tighter than the prior *and* still brackets the truth:

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(8, 4))
ax.hist(emu_prior["forecast"], facecolor="0.5", alpha=0.5, density=True, label="prior forecast")
ax.hist(emu_post["forecast"], facecolor="b", alpha=0.5, density=True, label="posterior forecast")
ylim = ax.get_ylim()
ax.plot([truth["forecast"]] * 2, ylim, "r", lw=3, label="truth")
ax.set_xlabel("forecast (peak SO$_4$, toy units)")
ax.set_ylabel("density")
ax.legend(loc="upper left")

That is data-space inversion in one figure. We never ran the model inside the loop, never touched a parameter field, never computed a Jacobian. We learned the prior covariance once, built a one-line emulator from it, and conditioned ten thousand cheap realisations on the data. The posterior forecast is sharper than the prior and still honest about the truth.

For DIZON this is the difference between an answer in seconds and an answer in days - which, given the 6-minute run cost, is often the difference between an answer and no answer at all.

## What DSI can and cannot answer

DSI is powerful precisely because it is narrow. Being clear about its boundaries is what keeps it honest.

**It can:**

- Produce a **posterior distribution of any forecast that was in the training ensemble**, conditioned on any subset of the measurements that were in the training ensemble. Peak SO₄ at the supply well, conditioned on history-period SO₄/O₂/NO₃/pH/Tmp - exactly our case.
- Do this for *many* forecasts and *many* conditioning sets at almost no extra cost, because the expensive part (the prior MC) is already spent. This is what makes the dataworth analysis - retraining on obs subsets to ask "would measuring the cations have helped?" - nearly free.

**It cannot:**

- Tell you about a **parameter field**. DSI never estimates hydraulic conductivity, porosity or pyrite abundance. If you need the posterior *parameters* - to map where the aquifer is permeable, say - DSI is the wrong tool; you need full-model history matching.
- Forecast a quantity, or honour a condition, that the **training ensemble did not span**. The emulator can only interpolate within the covariance it learned. Ask it about a supply-well pumping rate the prior never sampled, or a forecast at a time no realisation output, and it is extrapolating blind. This is the deep reason the optimization notebook needs its own *training sweep*: decision-variable coverage has to be designed into the ensemble, not assumed - we foreshadow it now so it is not a surprise later.

A one-sentence pointer for completeness: a nonlinear extension, **DSIAE** (DSI with an autoencoder, `pyemu.emulators.DSIAE`), replaces the linear SVD step with a learned encoder/decoder for ensembles whose structure a Gaussian cannot capture; it is a later add-on in this series, and the linear `DSI` is our workhorse.

## The series principle: fidelity check before conditioning, always

The toy emulator above was guaranteed to work - we built the ensemble from a Gaussian, so the linear emulator is exact. A real RTM ensemble offers no such guarantee. The covariance is estimated from a finite, transformed, non-Gaussian sample, and the emulator is only as good as that approximation.

So the non-negotiable beat in this series, before *any* emulator is used to condition or to inform a decision, is the **fidelity check**: take realisations the emulator never saw during training, predict their outputs with the emulator, and compare against the full model's actual outputs for those same realisations. If the emulator cannot reproduce held-out truth, it has no business producing a posterior forecast.

The check is nearly free, because the prior-MC runs the emulator was *not* trained on already exist - we simply hold some back. **Never trust an emulator you have not tested.** Every DSI notebook in part1 opens with this check, and you should treat it as mandatory in your own work.

## What the real thing looks like

You will not hand-roll the SVD in practice. `pyemu` packages all of it in the `DSI` emulator class. The toy machinery above maps onto it directly: `data` is the prior-MC observation ensemble (with the held-out truth dropped), `transforms` is the transform menu we discussed, and `energy_threshold` is the fraction of variance the truncated SVD keeps. Sketched, with no model runs:

```python
from pyemu.emulators import DSI

# data = prior-MC observation ensemble, truth realisation dropped
transforms = [{"type": "standard_scaler"}]
dsi = DSI(data=data, transforms=transforms, energy_threshold=0.975)
dsi.fit()                       # builds d_bar and Cd^{1/2} under the transform
dsi.latent_dim                  # how many latent parameters survived the truncation

# stand up a PEST++ run directory whose forward model IS the emulator
dpst = dsi.prepare_pestpp("dsi_template", use_runstor=True)
```

After `prepare_pestpp`, conditioning is an ordinary PESTPP-IES run - except every "model run" is the one-line emulator, so the whole history match finishes in seconds. We do all of this for real, on the DIZON prior ensemble, in [part1_05_dsi_basics](../part1_05_dsi_basics/dizon_dsi_basics.ipynb). For the underlying method see [Sun and Durlofsky (2017)](https://doi.org/10.1007/s11004-016-9672-8) and [Lima et al. (2020)](https://doi.org/10.1007/s10596-020-09933-w), and the GMDSI [overview of DSI by John Doherty](https://youtu.be/s2g3HaJa1Wk).

## Key points

- DSI works in **observation space**: it conditions model *outputs* on data without ever searching the high-dimensional space of model *parameters*. That sidesteps the dimensionality that makes full-model history matching expensive.
- The emulator is built from the prior ensemble's mean $\bar{\mathbf{d}}$ and covariance $\mathbf{C}_d$; an SVD turns $\mathbf{C}_d$ into a small **latent space** of standard-normal super-parameters, and a forward run is the one-liner $\mathbf{d}_{\text{PCA}} = \bar{\mathbf{d}} + \mathbf{C}_d^{1/2}\mathbf{x}$.
- Because the emulator is essentially free to run, conditioning (and dataworth, and optimization) become cheap once the prior MC is paid for - the whole rationale given the ~6 min/run DIZON cost.
- DSI answers **forecasts conditioned on data**; it does not recover parameter fields, and it cannot extrapolate beyond what the training ensemble spanned - which is why decision-space coverage must be *designed* (the training-sweep idea).
- Real RTM outputs are non-Gaussian, so the production `DSI` applies a **transform** before the SVD; the linear `DSI` is the workhorse and `DSIAE` is a nonlinear add-on for later.
- **Always run the fidelity check before conditioning.** Never trust an emulator you have not tested against held-out truth.